# Data Preparation Evaluation

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
import shutil
import zipfile


import warnings
warnings.filterwarnings('ignore')


output_dir = f"/content/drive/MyDrive/logo_recognition_similarity_search_project/output"
n_components=600
ds_path = f"{output_dir}/final-logo-recognition-with-{n_components}-hdbscan-category.tsv"
images_category_dir = f"{output_dir}/images_category"
images_category_zip = f"{images_category_dir}/images_category.zip"
print(ds_path)

/content/drive/MyDrive/logo_recognition_similarity_search_project/output/final-logo-recognition-with-600-hdbscan-category.tsv


In [13]:
df = pd.read_csv(ds_path,sep="\t")
df.head()

,id,path,prompt,category,path_category
0,1104391803911815278,images/00/00/1104391803911815278_0_0.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
1,1104391803911815278,images/00/00/1104391803911815278_0_1.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
2,1104391803911815278,images/00/00/1104391803911815278_1_0.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
3,1104391803911815278,images/00/00/1104391803911815278_1_1.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
4,1132464139621646377,images/00/00/1132464139621646377_0_0.png,"food truck, sells french fries, logo on side i...",127006,images_category/category_127006/11324641396216...


In [14]:
reduced_embeddings = np.load(f"{output_dir}/reduced_{n_components}_embeddings.npy")
reduced_embeddings.shape

(1777584, 600)

In [15]:
df.reset_index(inplace=True)
df.head()

,index,id,path,prompt,category,path_category
0,0,1104391803911815278,images/00/00/1104391803911815278_0_0.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
1,1,1104391803911815278,images/00/00/1104391803911815278_0_1.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
2,2,1104391803911815278,images/00/00/1104391803911815278_1_0.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
3,3,1104391803911815278,images/00/00/1104391803911815278_1_1.png,cute simple baby shark logo for company,197906,images_category/category_197906/11043918039118...
4,4,1132464139621646377,images/00/00/1132464139621646377_0_0.png,"food truck, sells french fries, logo on side i...",127006,images_category/category_127006/11324641396216...


In [16]:
from sklearn.metrics.pairwise import cosine_similarity

# Function to compute cosine similarity
def compute_similarity(index1, index2):
    embedding1 = reduced_embeddings[index1].reshape(1, -1)
    embedding2 = reduced_embeddings[index2].reshape(1, -1)
    return cosine_similarity(embedding1, embedding2)[0][0]

In [17]:
compute_similarity(1,2),compute_similarity(1,100)

(0.9995661, -0.57745516)

In [18]:
import random

# Set random seed for reproducibility
random.seed(101)
np.random.seed(101)

# Select categories that contain at least two logos
valid_categories = df['category'].value_counts()
valid_categories = valid_categories[valid_categories >= 2].index.tolist()

# Randomly select 100 categories from valid ones
if len(valid_categories) < 100:
    raise ValueError("Not enough valid categories with at least two logos.")
selected_categories = random.sample(valid_categories, 100)


# List to store logo pairs and similarity scores
logo_pairs = []
similarity_scores = []

# Process each selected category
for category in selected_categories:
    # Filter logos for the selected category
    category_logos = df[df['category'] == category]

    # Randomly select two logos
    selected_logos = category_logos.sample(n=2)

    # Retrieve indices from df
    index1 = selected_logos.iloc[0]['index']
    index2 = selected_logos.iloc[1]['index']

    # Compute similarity
    similarity = compute_similarity(index1, index2)
    similarity_scores.append(similarity)

    # Store results
    logo_pairs.append((
                       category,
                       selected_logos.iloc[0]['path_category'],
                       selected_logos.iloc[1]['path_category'],
                       similarity))

# Convert results to DataFrame
results_df = pd.DataFrame(logo_pairs, columns=['Category','Logo1', 'Logo2','Similarity'])
results_df

,Category,Logo1,Logo2,Similarity
0,57026,images_category/category_57026/110049451416408...,images_category/category_57026/110930288125911...,0.998482
1,183185,images_category/category_183185/11061784031131...,images_category/category_183185/11172749270542...,0.999511
2,93797,images_category/category_93797/111062884702565...,images_category/category_93797/110438705096006...,0.999109
3,180916,images_category/category_180916/10857990954798...,images_category/category_180916/10809612258543...,0.999777
4,185500,images_category/category_185500/11403743338108...,images_category/category_185500/11321140456049...,0.999721
...,...,...,...,...
95,41430,images_category/category_41430/108692990948357...,images_category/category_41430/109421683629753...,0.997545
96,65116,images_category/category_65116/112613874518853...,images_category/category_65116/108769069917406...,0.998084
97,143630,images_category/category_143630/10931260400517...,images_category/category_143630/10931260400517...,0.999699
98,43831,images_category/category_43831/114040398060271...,images_category/category_43831/110890382160378...,0.996764


In [19]:
# Compute percentage of similar logo pairs (threshold ≥ 0.8)
threshold = 0.8
similar_pairs = sum(results_df['Similarity'] >= threshold)
total_pairs = len(results_df)
similarity_percentage = (similar_pairs / total_pairs) * 100

print(f"Similarity percentage: {similarity_percentage:.2f}%")

Similarity percentage: 100.00%


In [20]:
# Save results
results_df.to_csv(f"{output_dir}/evaluation_{n_components}_hdbscan_category.tsv",sep="\t",index=False)



---



In [21]:
results_df = pd.read_csv(f"{output_dir}/evaluation_{n_components}_hdbscan_category.tsv",sep="\t")
results_df

,Category,Logo1,Logo2,Similarity
0,57026,images_category/category_57026/110049451416408...,images_category/category_57026/110930288125911...,0.998482
1,183185,images_category/category_183185/11061784031131...,images_category/category_183185/11172749270542...,0.999511
2,93797,images_category/category_93797/111062884702565...,images_category/category_93797/110438705096006...,0.999109
3,180916,images_category/category_180916/10857990954798...,images_category/category_180916/10809612258543...,0.999777
4,185500,images_category/category_185500/11403743338108...,images_category/category_185500/11321140456049...,0.999721
...,...,...,...,...
95,41430,images_category/category_41430/108692990948357...,images_category/category_41430/109421683629753...,0.997545
96,65116,images_category/category_65116/112613874518853...,images_category/category_65116/108769069917406...,0.998084
97,143630,images_category/category_143630/10931260400517...,images_category/category_143630/10931260400517...,0.999699
98,43831,images_category/category_43831/114040398060271...,images_category/category_43831/110890382160378...,0.996764


In [22]:
import zipfile
import os

os.makedirs('/content/images_category',exist_ok=True)

with zipfile.ZipFile(images_category_zip, 'r') as zip_ref:
    zip_ref.extractall('/content/images_category')


In [23]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def display_logo_pairs(results_df, n_rows=10):
  for index, row in results_df.sample(n_rows,random_state=42).iterrows():
    logo1_path = row['Logo1']
    logo2_path = row['Logo2']

    logo1 = mpimg.imread(logo1_path)
    logo2 = mpimg.imread(logo2_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(logo1)
    axes[0].set_title("Logo1")
    axes[0].axis('off')
    axes[1].imshow(logo2)
    axes[1].set_title("Logo2")
    axes[1].axis('off')
    axes[2].text(0.5, 0.5, f"Category: {row['Category']}\nSimilarity: {row['Similarity']}", ha='center', va='center', fontsize=12)
    axes[2].axis('off')
    plt.show()

In [25]:
display_logo_pairs(results_df,n_rows=10)

Output hidden; open in https://colab.research.google.com to view.